# Week 5 · Demo — Building a Golden Set
### The 20 questions that become every future week's quality baseline

Yesterday's demo showed *why* you can't test LLM answers with `assert` — you evaluate **properties**, not strings. This notebook builds the thing that makes property-based evaluation possible: a **golden set** — a curated list of questions, each paired with what a great answer looks like.

**How to use this notebook:** run top to bottom, then do the experiments at the end (write your own entry, break the validator). It's fully offline — **no API key needed** — because a golden set is *data*, not a model call.

**Where this goes in your app:** this notebook is the seed of `src/eval/golden.py` (the loader/validator) and `data/golden_set.jsonl` (your 20 entries).

---
### The scenario (continued from yesterday)
Same HR assistant, same policy (*remote work: up to 3 days/week with manager approval*). Yesterday we hard-coded "3 days" and "manager" inside a Python function — which we admitted **collapses at 20 questions**. The golden set is the fix: the facts move *out* of code and *into* data, one entry per question.

## 1 · Anatomy of a golden entry

An entry is a small record with four core fields (plus one optional field we'll use for the judge):

| field | what it is | example |
|---|---|---|
| `id` | stable identifier, `g001`–`g020` | `"g001"` |
| `question` | a **real user-style** question | `"How many days can I work remotely?"` |
| `ideal_answer` | what a great answer looks like | `"Up to 3 days per week, with manager approval."` |
| `notes` | anything a human/judge should know | `"happy-path"` |
| `must_mention` *(optional)* | facts the answer must contain | `["3 days", "manager approval"]` |

Let's make one as a plain Python dict first, to see the shape.

In [1]:
entry = {
    "id": "g001",
    "question": "How many days can I work remotely?",
    "ideal_answer": "Employees may work remotely up to 3 days per week, with manager approval.",
    "notes": "happy-path — the single most common question",
    "must_mention": ["3 days", "manager approval"],
}

for field, value in entry.items():
    print(f"{field:13}: {value}")

id           : g001
question     : How many days can I work remotely?
ideal_answer : Employees may work remotely up to 3 days per week, with manager approval.
notes        : happy-path — the single most common question
must_mention : ['3 days', 'manager approval']


That's the whole idea. The facts we hard-coded in yesterday's `is_accurate` (the "3 days", the "manager") now live in **data** — in `ideal_answer` and `must_mention` — instead of in Python. That's what lets one evaluator work across *any* question: it reads the facts from the entry rather than having them baked in.

## 2 · Storing entries as JSONL

We store the set as **JSONL** — "JSON Lines" — which means **one JSON object per line**, not a single big JSON array. Compare:

```
# JSON (one array)                    # JSONL (one object per line)
[                                     {"id": "g001", ...}
  {"id": "g001", ...},                {"id": "g002", ...}
  {"id": "g002", ...}                 {"id": "g003", ...}
]
```

Why JSONL wins for a growing dataset:
- **Append a new entry** = add one line. No need to parse-and-rewrite the whole file.
- **Git diffs stay clean** — adding entry 21 changes exactly one line, so code review shows exactly what you added.
- **Stream-friendly** — you can read it line by line without loading the whole file into memory.

Let's write three entries to a real file.

In [2]:
import json

entries = [
    {"id": "g001",
     "question": "How many days can I work remotely?",
     "ideal_answer": "Employees may work remotely up to 3 days per week, with manager approval.",
     "notes": "happy-path",
     "must_mention": ["3 days", "manager approval"]},
    {"id": "g015",
     "question": "I'm on a hybrid team — how do my remote days fit with core collaboration hours?",
     "ideal_answer": "You may take up to 3 remote days per week, but you must still be available "
                     "during the 11am-3pm core collaboration hours on those days.",
     "notes": "harder — requires combining the remote-work section AND the core-hours section",
     "must_mention": ["3", "core", "hours"]},
    {"id": "g019",
     "question": "Can I work remotely from another country for a month?",
     "ideal_answer": "The handbook doesn't cover international remote work. Please contact HR for guidance.",
     "notes": "edge — answer is NOT in the docs; the system should refuse gracefully, not invent a policy",
     "must_mention": ["contact HR"]},
]

with open("golden_set_sample.jsonl", "w", encoding="utf-8") as f:
    for e in entries:
        f.write(json.dumps(e) + "\n")

print("wrote", len(entries), "entries. Raw file contents:\n")
print(open("golden_set_sample.jsonl", encoding="utf-8").read())

wrote 3 entries. Raw file contents:

{"id": "g001", "question": "How many days can I work remotely?", "ideal_answer": "Employees may work remotely up to 3 days per week, with manager approval.", "notes": "happy-path", "must_mention": ["3 days", "manager approval"]}
{"id": "g015", "question": "I'm on a hybrid team \u2014 how do my remote days fit with core collaboration hours?", "ideal_answer": "You may take up to 3 remote days per week, but you must still be available during the 11am-3pm core collaboration hours on those days.", "notes": "harder \u2014 requires combining the remote-work section AND the core-hours section", "must_mention": ["3", "core", "hours"]}
{"id": "g019", "question": "Can I work remotely from another country for a month?", "ideal_answer": "The handbook doesn't cover international remote work. Please contact HR for guidance.", "notes": "edge \u2014 answer is NOT in the docs; the system should refuse gracefully, not invent a policy", "must_mention": ["contact HR"]}


Look at the raw file: **three lines, three complete JSON objects.** That's JSONL. Adding a fourth question means appending a fourth line — nothing else moves.

## 3 · Loading and validating — the seed of `src/eval/golden.py`

Reading the file back isn't just `json.load` — we want to **validate** each entry as we load it, so a malformed golden set fails *loudly* now rather than silently poisoning every eval run later. We use a Pydantic model (the same validation tool from Weeks 2 and 4) to define what a valid entry *is*.

In [3]:
from pydantic import BaseModel, Field

class GoldenEntry(BaseModel):
    id: str
    question: str = Field(min_length=1)         # must be non-empty
    ideal_answer: str = Field(min_length=1)     # must be non-empty
    notes: str = ""                             # optional, defaults to empty
    must_mention: list[str] = []               # optional list of required facts

def load_golden(path):
    """Read a .jsonl file, validate every line, return a list of GoldenEntry."""
    out = []
    with open(path, encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue                        # skip blank lines
            data = json.loads(line)             # str -> dict
            out.append(GoldenEntry(**data))     # dict -> validated model (raises if invalid)
    return out

golden = load_golden("golden_set_sample.jsonl")
print(f"loaded {len(golden)} valid entries\n")
for e in golden:
    print(f"{e.id}: {e.question[:55]}...")

loaded 3 valid entries

g001: How many days can I work remotely?...
g015: I'm on a hybrid team — how do my remote days fit with c...
g019: Can I work remotely from another country for a month?...


`load_golden` does three jobs per line: **strip** whitespace, **skip** blanks, **parse** the JSON to a dict, and **validate** it into a `GoldenEntry`. If any line is broken, it fails at load time with a precise error — which is exactly what you want. Let's prove the validation actually bites.

In [4]:
# a deliberately broken entry: empty question (violates min_length=1)
bad_line = json.dumps({"id": "gBAD", "question": "", "ideal_answer": "something"})
print("bad entry:", bad_line, "\n")

from pydantic import ValidationError
try:
    GoldenEntry(**json.loads(bad_line))
except ValidationError as e:
    print("REJECTED as expected:\n", e)

bad entry: {"id": "gBAD", "question": "", "ideal_answer": "something"} 

REJECTED as expected:
 1 validation error for GoldenEntry
question
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.8/v/string_too_short


The validator refuses the entry and tells you *which field* broke *which rule* (`question` — String should have at least 1 character). A golden set that loads is a golden set you can trust; one that silently accepts junk is worse than none.

## 4 · The *mix* matters as much as the count

A golden set of 20 easy questions is a **mirror**, not a **measure** — it'll tell you your system is wonderful right up until real users arrive. A representative set mixes three kinds, roughly **14 / 4 / 2**:

In [5]:
# label our three sample entries by kind (in practice you'd track this in notes)
kinds = {
    "g001": "HAPPY  — everyday question, answer is directly in the docs",
    "g015": "HARDER — multi-hop, needs two sections combined",
    "g019": "EDGE   — answer NOT in the docs; must refuse gracefully",
}
for e in golden:
    print(f"{e.id}  [{kinds[e.id]}]")
    print(f"        ideal: {e.ideal_answer[:70]}...")
    print()

print("Target mix for 20 entries:  ~14 happy  ·  ~4 harder  ·  ~2 edge")

g001  [HAPPY  — everyday question, answer is directly in the docs]
        ideal: Employees may work remotely up to 3 days per week, with manager approv...

g015  [HARDER — multi-hop, needs two sections combined]
        ideal: You may take up to 3 remote days per week, but you must still be avail...

g019  [EDGE   — answer NOT in the docs; must refuse gracefully]
        ideal: The handbook doesn't cover international remote work. Please contact H...

Target mix for 20 entries:  ~14 happy  ·  ~4 harder  ·  ~2 edge


The **edge case (g019) is the most important entry in the set.** Its `ideal_answer` is a graceful *"I don't know — contact HR."* This is where you catch **hallucination**: a RAG system that invents an international-remote-work policy instead of admitting the docs don't cover it is *dangerous*, and only an edge entry with an honest ideal answer will expose it. Every golden set needs at least one "the correct answer is *I don't have that*."

The `must_mention` field is your bridge to tomorrow's judge: for g019 it's `["contact HR"]`, so the judge can check the answer points the user to HR rather than fabricating a policy.

## 5 · Sizing and sourcing (the rules that decide if any of this is worth anything)

**Size:**
- **20** — the floor. Enough to be *useful*, barely.
- **50** — genuinely *trustworthy*.
- **100+** — *serious* evaluation.

Start at 20; grow it as you go.

**Sourcing — in strict order of preference:**
1. **Real user questions** — BEST. If you have query logs, mine them.
2. **A domain expert imagining questions** — GOOD.
3. **You, at the keyboard, writing honestly** — OK (include ones you're *not sure* your system handles).
4. **Questions curated to make your system look good** — **AVOID.** This is the cardinal sin.

> **Garbage golden set = garbage signal forever after.** These 20 questions are what *every later week's KPI* is computed against. A soft, flattering set turns every number you ever report into a comforting lie. Spend a real 45 minutes here — don't churn it out in 15.

## 6 · Your turn — experiment

1. **Write a 4th entry** (`g002`) — an easy happy-path question from *your* capstone domain. Add it to `entries`, re-run the write + load cells, confirm it loads.
2. **Break the validator on purpose** — write a line missing `ideal_answer`, or with `must_mention` as a string instead of a list. Watch the `ValidationError` name the exact problem.
3. **Add a second edge case** — a question whose honest answer is "that's not in the documents." These are the entries that catch hallucination; you want at least two.
4. **Try a properties-only ideal** — for a question where the full answer depends on context, set `ideal_answer` to a short spec like *"Must state the 3-day limit and mention manager approval; under 40 words."* Both a full answer and a properties spec are valid `ideal_answer` styles.

## Fit it into your app

Two concrete artifacts come straight out of this notebook:

- **`data/golden_set.jsonl`** — your real 20 entries (Lab Step 2). Same JSONL format you just wrote; your domain, your questions, the 14/4/2 mix.
- **`src/eval/golden.py`** — a cleaned-up `load_golden` plus the `GoldenEntry` model. The loader you wrote here *is* that file, minus the demo scaffolding.

Then tomorrow, `src/eval/judge.py` takes each `GoldenEntry`, sends its `question` to your `/ask` endpoint, and scores the candidate answer against the entry's `ideal_answer` and `must_mention`. The golden set is the fuel; the judge is the engine.

**The one-line takeaway:** *move the facts out of your code and into data — one honest, representative entry per question — because that data is what every future quality number depends on.*